# ShopSphere Purchase Journey Analytics: Exploratory Data Analysis
**Analyst:** Product Analyst / Business Analyst  
**Dataset:** Synthetic Clickstream Event Stream (`N = 120,000` sessions, `689,508` events)  
**Master Seed:** Fixed `SEED = 42` (100% Reproducible)  

---
## 1. Environment Initialization & Data Ingestion

In [1]:
import duckdb
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

# Ingest processed Parquet tables
df_customers = pd.read_parquet('../data/processed/customers.parquet')
df_products = pd.read_parquet('../data/processed/products.parquet')
df_sessions = pd.read_parquet('../data/processed/sessions.parquet')
df_events = pd.read_parquet('../data/processed/events.parquet')

con = duckdb.connect()
con.register('customers', df_customers)
con.register('products', df_products)
con.register('sessions', df_sessions)
con.register('events', df_events)

print(f'Ingested: {len(df_sessions):,} sessions, {len(df_events):,} events, {len(df_customers):,} customers, {len(df_products):,} products.')

## 2. Macro Funnel Progression & Conversion Drop-Offs
Evaluating step-by-step conversion across the 11-stage purchase journey.

In [2]:
with open('../sql/01_funnel_analysis.sql') as f:
    sql_funnel = f.read()
df_funnel = con.execute(sql_funnel).fetchdf()
display(df_funnel)

## 3. Journey Timing & Checkout Dwell Times
Analyzing non-parametric duration distributions across stages.

In [3]:
with open('../sql/02_journey_timing.sql') as f:
    sql_timing = f.read()
# Execute individual statements
statements = [s.strip() for s in sql_timing.split(';') if s.strip()]
for i, stmt in enumerate(statements, 1):
    print(f'--- Query Part {i} ---')
    display(con.execute(stmt).fetchdf())

## 4. Cross-Device Funnel Disparities
Evaluating conversion and checkout friction across Mobile, Desktop, and Tablet.

In [4]:
with open('../sql/03_device_analysis.sql') as f:
    sql_dev = f.read()
df_dev = con.execute(sql_dev).fetchdf()
display(df_dev)

## 5. Statistical Hypothesis Testing Suite (H1–H10)
Executing formal inferential tests (Chi-Square, Mann-Whitney U, Logistic Regressions).

In [5]:
# Formal logistic regression on mobile address drop-off (H1)
addr_events = df_events[df_events['event_type'] == 'address_entry']['session_id'].unique()
ship_events = df_events[df_events['event_type'] == 'shipping_view']['session_id'].unique()
df_addr = df_sessions[df_sessions['session_id'].isin(addr_events)].copy()
df_addr['passed_address'] = df_addr['session_id'].isin(ship_events)

logit_h1 = smf.logit("passed_address ~ C(device_type, Treatment('desktop')) + C(customer_type)", data=df_addr).fit()
print(logit_h1.summary())

## 6. Payment Recovery & Retry Diagnostics
Quantifying payment failure rates, recovery pathways, and method switching.

In [6]:
with open('../sql/09_payment_recovery.sql') as f:
    sql_pay = f.read()
statements = [s.strip() for s in sql_pay.split(';') if s.strip()]
for i, stmt in enumerate(statements, 1):
    print(f'--- Payment Diagnostics Part {i} ---')
    display(con.execute(stmt).fetchdf())